# Task 3 — Manual (Human) Evaluation

We load the `assignment_01.xlsx` produced in Task 2, add a cost column, manually rate 10–15 products, compute final scores, and analyze which criteria performed best and worst.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from shared.config import GENERATION_MODEL, OUTPUT_EXCEL
from shared.constants import CRITERIA, JUDGE_CRITERIA, RUBRIC, final_score
from shared.utils import calculate_cost

In [ ]:
df = pd.read_excel(OUTPUT_EXCEL)
print(f"{len(df)} products loaded")
df.head(3)

## 1. Cost Calculation

Convert `input_tokens` and `output_tokens` to USD using the per-1K-token pricing for Gemma-2-9b-it. Note: input and output tokens can be priced differently.

In [ ]:
df["cost_usd"] = df.apply(
    lambda row: calculate_cost(GENERATION_MODEL, row["input_tokens"], row["output_tokens"]),
    axis=1,
)

print(f"Total cost: ${df['cost_usd'].sum():.6f}")
print(f"Avg cost per call: ${df['cost_usd'].mean():.6f}")
df[["product_name", "input_tokens", "output_tokens", "cost_usd"]].head()

## 2. Rate Each Criterion (10–15 products)

We select 15 products (evenly spaced across the dataset) and manually rate each criterion as good / ok / bad according to the rubric from Task 1.

Latency and Cost are rated programmatically based on the thresholds in the rubric.

In [ ]:
# Select 15 evenly spaced indices for manual evaluation
import numpy as np

EVAL_COUNT = 15
eval_indices = np.linspace(0, len(df) - 1, EVAL_COUNT, dtype=int).tolist()
print(f"Products selected for evaluation (indices): {eval_indices}")
print()
for i in eval_indices:
    row = df.iloc[i]
    word_count = len(row["generated_description"].split())
    print(f"--- [{i}] {row['product_name']} ({word_count} words) ---")
    print(row["generated_description"])
    print()

In [ ]:
def rate_latency(latency_ms: float) -> str:
    if latency_ms <= 2000:
        return "good"
    if latency_ms <= 5000:
        return "ok"
    return "bad"


def rate_cost(cost_usd: float) -> str:
    if cost_usd <= 0.001:
        return "good"
    if cost_usd <= 0.005:
        return "ok"
    return "bad"


def rate_length(text: str) -> str:
    wc = len(text.split())
    if 50 <= wc <= 90:
        return "good"
    if 40 <= wc <= 110:
        return "ok"
    return "bad"

In [ ]:
# ============================================================
# MANUAL RATINGS — fill in after reading each description above
# Keys: index -> {criterion: rating}
# Only Fluency, Grammar, Tone, Grounding are truly manual.
# Length, Latency, Cost are computed automatically.
# ============================================================

manual_ratings: dict[int, dict[str, str]] = {
    # Example (replace with your actual ratings):
    # 0: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    # 3: {"Fluency": "ok",   "Grammar": "good", "Tone": "ok",   "Grounding": "bad"},
}

assert len(manual_ratings) >= 10, (
    f"Rate at least 10 products. Currently rated: {len(manual_ratings)}"
)

## 3. Apply Final Score

In [ ]:
for idx, human_scores in manual_ratings.items():
    row = df.iloc[idx]
    full_ratings = {
        **human_scores,
        "Length": rate_length(row["generated_description"]),
        "Latency": rate_latency(row["latency_ms"]),
        "Cost": rate_cost(row["cost_usd"]),
    }
    for criterion, rating in full_ratings.items():
        df.at[idx, criterion] = rating
    df.at[idx, "final_score"] = final_score(full_ratings)

evaluated = df[df["final_score"] != ""]
print(f"Evaluated {len(evaluated)} products")
evaluated[["product_name"] + CRITERIA + ["final_score"]]

In [ ]:
df.to_excel(OUTPUT_EXCEL, index=False)
print(f"Saved to {OUTPUT_EXCEL}")

## 4. Baseline Analysis

In [ ]:
rating_to_num = {"good": 2, "ok": 1, "bad": 0}

print("=== Pass / Fail Summary ===")
print(evaluated["final_score"].value_counts())
print(f"Pass rate: {(evaluated['final_score'] == 'pass').mean():.0%}")
print()

print("=== Rating Distribution per Criterion ===")
for c in CRITERIA:
    counts = evaluated[c].value_counts()
    print(f"\n{c}:")
    for rating in ["good", "ok", "bad"]:
        print(f"  {rating}: {counts.get(rating, 0)}")

print()
print("=== Average Score per Criterion (good=2, ok=1, bad=0) ===")
avg_scores = {}
for c in CRITERIA:
    numeric = evaluated[c].map(rating_to_num)
    avg_scores[c] = numeric.mean()

sorted_criteria = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)
for c, score in sorted_criteria:
    print(f"  {c}: {score:.2f}")

print(f"\nBest performing:  {sorted_criteria[0][0]} ({sorted_criteria[0][1]:.2f})")
print(f"Worst performing: {sorted_criteria[-1][0]} ({sorted_criteria[-1][1]:.2f})")

## Baseline Analysis Summary

_Fill in after running the cells above:_

- **Best criterion:** (e.g., Grammar — most descriptions had correct spelling/punctuation)
- **Worst criterion:** (e.g., Length — model frequently exceeded 90 words)
- **Pass rate:** X%
- **Key observations:** (what patterns do you see? which failures are most common?)
- **Improvement strategy for Task 4:** (based on the weakest criteria, what would you change?)